# Building a Character-Level Language Model with LSTMs

## What is this Notebook?
In this notebook, we are shifting our focus from predicting continuous numerical values (regression) to understanding and generating human language. We will build a **Character-Level Language Model** using a Long Short-Term Memory (LSTM) network. 

Instead of treating text as words, this model operates on the most fundamental level: individual characters (letters, punctuation marks, and spaces). By observing a sequence of characters, the network learns to predict which specific character is most likely to appear next.

## The General Working Principle
At its core, a language model is a probability engine. The architecture follows a specific pipeline to process discrete text:

1. **Tokenization (Character-to-Integer):** Neural networks cannot read letters. We first identify every unique character in our dataset (the vocabulary) and assign each a unique integer ID.
2. **The Embedding Layer:** Unlike raw numbers, these integer IDs represent categories. The Embedding layer transforms these IDs into dense mathematical vectors, allowing the network to learn relationships between different characters.
3. **The LSTM Core:** The sequence of vectors is fed into the LSTM. The network uses its hidden state to remember the historical context (e.g., "I just saw the letters 't' and 'h', so the next letter is probably 'e' or 'a'").
4. **Classification:** The output of the LSTM is passed through a Linear layer, which scores every possible character in our vocabulary. We use these scores to calculate the probability of the next character.

## The Data
We are using raw, unstructured text data-specifically, the "Tiny Shakespeare" dataset, which contains a collection of the playwright's works. 

Unlike standard datasets that come with clear features and labels, we have to create our own. We use a **Sliding Window** approach: we take a chunk of text (e.g., 100 characters) as our input ($X$), and the exact same chunk shifted one character to the right as our target ($Y$). This transforms a massive wall of text into a supervised learning problem.

## The Goal
Our objective is to train the LSTM to minimize the **Cross-Entropy Loss**. Once the loss is low enough, the model will have internalized the fundamental rules of English spelling, vocabulary, and basic grammar. We can then use the trained model for **Inference (Text Generation)**: feeding it a starting seed (a "prime" string) and asking it to write a completely new, original text character by character.

## Intuition: How does it learn?
Think of this model as an incredibly advanced smartphone auto-complete. When initialized, the model doesn't know what a word is, what a space is, or even what a vowel is. 

Through training, the LSTM's memory cells learn complex patterns. In the short term, it learns basic spelling (if it sees `q`, it expects a `u`). In the medium term, it learns words (if it sees `k n i g h`, it expects a `t`). In the long term, it learns formatting—for example, if it opens a parenthesis, its memory cell will hold onto that context for dozens of steps until it generates a closing parenthesis.

In [42]:
import torch
from torch import nn
import numpy as np
import requests
import torch_directml
import torch.nn.functional as F

## Understanding the Hyperparameters

Before we process our data or build our model, we must define the hyperparameters that will dictate how our network learns and operates. 

* **`SEQ_LENGTH = 100`**: The context window. This defines how many characters the LSTM will look at simultaneously to predict the next character. A longer sequence allows the model to capture longer dependencies (like multi-syllable words or punctuation pairing), but takes more memory to train.
* **`BATCH_SIZE = 64`**: The number of sequences processed in parallel during one training step. 
* **`EMBEDDING_DIM = 128`**: The size of the mathematical vector used to represent each character. Instead of just a single integer, each character becomes a 128-dimensional point in space, allowing the network to learn similarities between different characters (e.g., grouping vowels together).
* **`HIDDEN_SIZE = 256`**: The memory capacity of our LSTM cells. A larger hidden size allows the network to remember more complex, nuanced patterns in the text, but increases computational cost and the risk of overfitting.
* **`NUM_LAYERS = 2`**: The depth of the network. Stacking two LSTMs on top of each other allows the model to learn hierarchical features (e.g., Layer 1 learns spelling, Layer 2 learns grammar).
* **`LEARNING_RATE = 1e-3`**: Controls how aggressively the optimizer updates the network's weights during training.
* **`NUM_EPOCHS = 20`**: The number of complete passes the training loop will make over the entire dataset.
* **`DEVICE = "cpu"`**: Defines the hardware accelerator used for matrix calculations. (Can be changed to `"cuda"` if a GPU is available).
* **`CLIP = 5`**: The threshold for *Gradient Clipping*. As mentioned in the introduction, LSTMs can suffer from exploding gradients. This parameter explicitly caps the maximum size of a gradient update to keep training stable.
* **`URL`**: The source link to our raw training data (in this case, the Tiny Shakespeare dataset).

In [43]:
SEQ_LENGTH = 100
BATCH_SIZE = 64
EMBEDDING_DIM = 128
HIDDEN_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
NUM_LAYERS = 2
DEVICE = "cpu"
CLIP = 5

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

## Data Loading and Tokenization

Unlike regression tasks where data is already numerical, language models require an extra preprocessing step to translate text into numbers. This process is called **Tokenization**. In this notebook, we are performing character-level tokenization.

Here is what the code achieves step-by-step:

* **Fetching and Truncating:** We download the raw text data from the provided URL. For the sake of faster training and prototyping in this notebook, we slice the text to use only the first 100,000 characters.
* **Building the Vocabulary:** By applying `set(text)`, we extract every unique character present in our dataset (including spaces, line breaks, and punctuation). The total number of these unique characters becomes our `VOCAB_SIZE`.
* **Mapping Dictionaries:** Neural networks only understand numbers. We create two crucial dictionaries:
    * `char2int`: Maps each unique character to an integer ID. This is used to encode our training data.
    * `int2char`: Maps the integer IDs back to characters. We will use this later to decode the model's predictions into readable text.
* **Encoding the Data:** Finally, we loop through the entire raw text and convert it into `encoded_text`, a single, massive NumPy array of integers ready to be sliced into training sequences.

In [44]:
response = requests.get(URL)
text = response.text
text = text[:100000]

chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: i for i, ch in int2char.items()}

encoded_text = np.array([char2int[ch] for ch in text])

VOCAB_SIZE = len(chars)

In [45]:
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr) // batch_size_total
    arr = arr[:n_batches * batch_size_total]

    arr = arr.reshape((batch_size, -1))

    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        y = np.zeros_like(x)

        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]

        yield x, y


batches = get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH)
x_sample, y_sample = next(batches)

In [46]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        lstm_out = lstm_out.reshape(-1, self.hidden_size)
        output = self.fc(lstm_out)

        return output, hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

model = CharLSTM(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(DEVICE)
print(model)

CharLSTM(
  (embedding): Embedding(61, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=61, bias=True)
)


In [47]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(NUM_EPOCHS):
    h = model.init_hidden(BATCH_SIZE, DEVICE)

    for x, y in get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH):
        inputs = torch.from_numpy(x).long().to(DEVICE)
        targets = torch.from_numpy(y).long().to(DEVICE)

        h = tuple([each.data for each in h])

        optimizer.zero_grad()
        output, h = model(inputs, h)
        loss = criterion(output, targets.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

    print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}")

Epoch: 1/20, Loss: 3.2686
Epoch: 2/20, Loss: 3.0475
Epoch: 3/20, Loss: 2.7401
Epoch: 4/20, Loss: 2.5080
Epoch: 5/20, Loss: 2.3642
Epoch: 6/20, Loss: 2.2626
Epoch: 7/20, Loss: 2.1796
Epoch: 8/20, Loss: 2.1059
Epoch: 9/20, Loss: 2.0380
Epoch: 10/20, Loss: 1.9812
Epoch: 11/20, Loss: 1.9265
Epoch: 12/20, Loss: 1.8826
Epoch: 13/20, Loss: 1.8377
Epoch: 14/20, Loss: 1.8056
Epoch: 15/20, Loss: 1.7663
Epoch: 16/20, Loss: 1.7365
Epoch: 17/20, Loss: 1.7107
Epoch: 18/20, Loss: 1.6823
Epoch: 19/20, Loss: 1.6562
Epoch: 20/20, Loss: 1.6364


In [48]:
def predict(model, char, h=None, top_k=None, temperature=1.0):
    model.eval()

    x = np.array([[char2int[char]]])
    input = torch.from_numpy(x).long().to(DEVICE)

    if h is not None:
        h = tuple([each.data for each in h])

    out, h = model(input, h)
    logits = out[-1] / temperature

    p = F.softmax(logits, dim=0).detach().cpu().numpy()

    if top_k is not None:
        p[np.argsort(p)[:-top_k]] = 0
        p = p / p.sum()

    char_index = np.random.choice(len(chars), p=p)

    return int2char[char_index], h

def generate(model, size=500, prime='The ', top_k=5, temperature=1.0):
    model.eval()

    chars_generated = [ch for ch in prime]
    h = model.init_hidden(1, DEVICE)

    for ch in prime:
        char, h = predict(model, ch, h, top_k=top_k, temperature=temperature)

    chars_generated.append(char)

    for _ in range(size - 1):
        char, h = predict(model, chars_generated[-1], h, top_k=top_k, temperature=temperature)
        chars_generated.append(char)

    return ''.join(chars_generated)

In [49]:
temperatures = [0.25, 0.5, 0.75, 1.0, 1.25]

for temp in temperatures:
    print(f"\nTemperature: {temp}")
    generated_text = generate(model, size=400, prime='KING:', top_k=5, temperature=temp)
    print(generated_text)


Temperature: 0.25
KING:
He have strove the people are the state the people,
I prain the people the people of the man the people.

MENENIUS:
The consul the people to the pears of the people,
And the war the people and the people as the serves the people,
And the stand the people and the people the pears,
Where he common the people are the senate
The speak the marter of the speat of the people the people,
And the man the 

Temperature: 0.5
KING:
He speat it the can the will the him a shord of the people
The stand the warth the common to to the consul
With he shollow the conger: and the words the wars and the common
That he speak, the pored, the peoples and the people.

CORIOLANUS:
I have shall stand the words and with he had the pearts there
The steed the pears and the man the common the consul
The past the speak his not to sure to the p

Temperature: 0.75
KING:
Wells, he would with hat me the mark is.

MARCIUS:
Shall the hath wite marked hind heard walls.

BRUTUS:
In we sell the people